# Primary readout

Conversion rate for the three pre-registered comparisons (Mens vs No E-Mail, Womens vs No E-Mail, Mens vs Womens), Holm corrected, followed by the spend readout. Methods are fixed in `docs/experiment_design.md` and `docs/Log.md` before this notebook was run:
- Wald (unpooled) CI as the main interval, Newcombe hybrid score as a cross-check
- CIs at alpha/3 (98.33%) as the main interval, 95% shown as context
- z-test uses the same unpooled SE as the CI, pooled SE shown once as a cross-check

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats
from load import load_hillstrom

df = load_hillstrom()
TABLES = Path.cwd().parent / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

print(f"rows: {len(df)}")

rows: 64000


## 0. Pre-check, are there enough conversions for the Wald interval?

The Wald interval leans on a normal approximation, which gets shaky when an arm has few events. Before splitting any outcome by arm, checking how many conversions each arm should have using only pooled numbers:

expected conversions in an arm = pooled conversion rate x arm size

The pooled rate is the same 0.9031% validated in `00_eda_pre_registration.ipynb`. Arm sizes are counts of `segment`. Neither splits an outcome by arm.

In [2]:
pooled_rate = df["conversion"].mean()
total_conversions = df["conversion"].sum()
arm_sizes = df["segment"].value_counts().sort_index()
expected_conversions = pooled_rate * arm_sizes
expected_non_conversions = (1 - pooled_rate) * arm_sizes

print(f"pooled conversion rate: {pooled_rate:.4%}")
print(f"total conversions (all arms pooled): {total_conversions:,}")
print()
pd.DataFrame({
    "arm_size": arm_sizes,
    "expected_conversions": expected_conversions,
    "expected_non_conversions": expected_non_conversions,
})

pooled conversion rate: 0.9031%
total conversions (all arms pooled): 578



,arm_size,expected_conversions,expected_non_conversions
segment,,,
Mens E-Mail,21307,192.428844,21114.571156
No E-Mail,21306,192.419812,21113.580187
Womens E-Mail,21387,193.151344,21193.848656


## 1. Conversion by arm

From here on outcomes are split by arm, which the pre-registration commits allow. For each arm, the customer count n, the number who converted, and the conversion rate p = conversions / n.

In [3]:
from statsmodels.stats.proportion import confint_proportions_2indep
from statsmodels.stats.multitest import multipletests

ALPHA = 0.05
N_COMPARISONS = 3
ALPHA_ADJ = ALPHA / N_COMPARISONS
MDE = 0.005

by_arm = df.groupby("segment")["conversion"].agg(n="count", conversions="sum")
by_arm["rate"] = by_arm["conversions"] / by_arm["n"]
by_arm["variance_term"] = by_arm["rate"] * (1 - by_arm["rate"]) / by_arm["n"]
by_arm.style.format({"rate": "{:.4%}", "variance_term": "{:.3e}"})

,n,conversions,rate,variance_term
segment,,,,
Mens E-Mail,21307,267,1.2531%,5.808e-07
No E-Mail,21306,122,0.5726%,2.672e-07
Womens E-Mail,21387,189,0.8837%,4.096e-07


## 2. One comparison worked by hand, Mens vs No E-Mail

difference = p_treatment - p_control

unpooled SE = sqrt( p_t(1-p_t)/n_t + p_c(1-p_c)/n_c ), the same SE the Wald interval uses

z = difference / SE, and the two-sided p-value = 2 x P(Z > |z|)

Wald interval = difference +/- z_crit x SE, with z_crit for 98.33% (main) and 95% (context)

In [4]:
t, c = by_arm.loc["Mens E-Mail"], by_arm.loc["No E-Mail"]

diff = t["rate"] - c["rate"]
se = np.sqrt(t["variance_term"] + c["variance_term"])
z = diff / se
p = 2 * stats.norm.sf(abs(z))
z_crit_adj = stats.norm.ppf(1 - ALPHA_ADJ / 2)
z_crit_95 = stats.norm.ppf(1 - ALPHA / 2)

print(f"p_t {t['rate']:.6f}, p_c {c['rate']:.6f}")
print(f"difference {diff:.6f} ({diff*100:.4f} pp)")
print(f"unpooled SE {se:.6f}")
print(f"z {z:.4f}")
print(f"raw two-sided p {p:.3e}")
print(f"z_crit 98.33% {z_crit_adj:.4f}, z_crit 95% {z_crit_95:.4f}")
print(f"98.33% Wald CI ({(diff - z_crit_adj*se)*100:.4f}, {(diff + z_crit_adj*se)*100:.4f}) pp")
print(f"95% Wald CI ({(diff - z_crit_95*se)*100:.4f}, {(diff + z_crit_95*se)*100:.4f}) pp")

p_t 0.012531, p_c 0.005726
difference 0.006805 (0.6805 pp)
unpooled SE 0.000921
z 7.3899
raw two-sided p 1.469e-13
z_crit 98.33% 2.3940, z_crit 95% 1.9600
98.33% Wald CI (0.4601, 0.9010) pp
95% Wald CI (0.5000, 0.8610) pp


### Cross-check, pooled SE

The pooled SE assumes H0 is true, so both arms share one rate p_pool = (conversions_t + conversions_c) / (n_t + n_c):

pooled SE = sqrt( p_pool(1-p_pool) x (1/n_t + 1/n_c) )

The readout uses the unpooled SE, this just shows how much the choice moves the result.

In [5]:
p_pool = (t["conversions"] + c["conversions"]) / (t["n"] + c["n"])
se_pooled = np.sqrt(p_pool * (1 - p_pool) * (1 / t["n"] + 1 / c["n"]))
z_pooled = diff / se_pooled

print(f"p_pool {p_pool:.6f}")
print(f"pooled SE {se_pooled:.6f} vs unpooled SE {se:.6f}, ratio {se_pooled / se:.4f}")
print(f"z pooled {z_pooled:.4f} vs z unpooled {z:.4f}")
print(f"p pooled {2 * stats.norm.sf(abs(z_pooled)):.3e} vs p unpooled {p:.3e}")

p_pool 0.009129
pooled SE 0.000921 vs unpooled SE 0.000921, ratio 1.0006
z pooled 7.3851 vs z unpooled 7.3899
p pooled 1.523e-13 vs p unpooled 1.469e-13


## 3. All three comparisons

Same formulas for every pre-registered comparison. Mens vs Womens is read as Mens minus Womens. The Newcombe hybrid score interval (98.33%) is computed beside the Wald interval as the cross-check.

In [6]:
comparisons = [
    ("Mens E-Mail", "No E-Mail"),
    ("Womens E-Mail", "No E-Mail"),
    ("Mens E-Mail", "Womens E-Mail"),
]

rows = []
for arm_t, arm_c in comparisons:
    t, c = by_arm.loc[arm_t], by_arm.loc[arm_c]
    diff = t["rate"] - c["rate"]
    se = np.sqrt(t["variance_term"] + c["variance_term"])
    z = diff / se
    newc_lo, newc_hi = confint_proportions_2indep(
        t["conversions"], t["n"], c["conversions"], c["n"], method="newcomb", compare="diff", alpha=ALPHA_ADJ
    )
    rows.append({
        "comparison": f"{arm_t} vs {arm_c}",
        "rate_t": t["rate"],
        "rate_c": c["rate"],
        "diff_pp": diff * 100,
        "relative_lift": diff / c["rate"],
        "se_pp": se * 100,
        "z": z,
        "p_raw": 2 * stats.norm.sf(abs(z)),
        "wald_lo_9833_pp": (diff - z_crit_adj * se) * 100,
        "wald_hi_9833_pp": (diff + z_crit_adj * se) * 100,
        "newcombe_lo_9833_pp": newc_lo * 100,
        "newcombe_hi_9833_pp": newc_hi * 100,
        "wald_lo_95_pp": (diff - z_crit_95 * se) * 100,
        "wald_hi_95_pp": (diff + z_crit_95 * se) * 100,
    })
readout = pd.DataFrame(rows)
readout[["comparison", "rate_t", "rate_c", "diff_pp", "relative_lift", "se_pp", "z", "p_raw"]].style.format(
    {"rate_t": "{:.4%}", "rate_c": "{:.4%}", "diff_pp": "{:.4f}", "relative_lift": "{:.2%}", "se_pp": "{:.4f}", "z": "{:.4f}", "p_raw": "{:.3e}"}
)

,comparison,rate_t,rate_c,diff_pp,relative_lift,se_pp,z,p_raw
0,Mens E-Mail vs No E-Mail,1.2531%,0.5726%,0.6805,118.84%,0.0921,7.3899,1.469e-13
1,Womens E-Mail vs No E-Mail,0.8837%,0.5726%,0.3111,54.33%,0.0823,3.7817,1.558e-04
2,Mens E-Mail vs Womens E-Mail,1.2531%,0.8837%,0.3694,41.80%,0.0995,3.7120,2.056e-04


In [7]:
readout[["comparison", "wald_lo_9833_pp", "wald_hi_9833_pp", "newcombe_lo_9833_pp", "newcombe_hi_9833_pp", "wald_lo_95_pp", "wald_hi_95_pp"]].round(4)

,comparison,wald_lo_9833_pp,wald_hi_9833_pp,newcombe_lo_9833_pp,newcombe_hi_9833_pp,wald_lo_95_pp,wald_hi_95_pp
0,Mens E-Mail vs No E-Mail,0.4601,0.9010,0.4619,0.9058,0.5000,0.8610
1,Womens E-Mail vs No E-Mail,0.1142,0.5080,0.1143,0.5116,0.1499,0.4723
2,Mens E-Mail vs Womens E-Mail,0.1312,0.6076,0.1313,0.6106,0.1744,0.5644


## 4. Holm correction, by hand

Holm sorts the m = 3 raw p-values from smallest to largest. The k-th smallest is multiplied by (m - k + 1), so by 3, then 2, then 1. To keep the adjusted values in the same order as the raw ones, each adjusted value is the running maximum of the ones before it, capped at 1. A comparison is significant if its adjusted p-value is below 0.05.

Cross-checked against statsmodels `multipletests(method="holm")`.

In [8]:
order = readout["p_raw"].sort_values().index
holm = pd.DataFrame({"comparison": readout.loc[order, "comparison"], "p_raw": readout.loc[order, "p_raw"]})
holm["multiplier"] = [N_COMPARISONS - k for k in range(N_COMPARISONS)]
holm["p_times_multiplier"] = holm["p_raw"] * holm["multiplier"]
holm["p_holm"] = holm["p_times_multiplier"].cummax().clip(upper=1)

_, p_holm_sm, _, _ = multipletests(readout["p_raw"], alpha=ALPHA, method="holm")
holm["p_holm_statsmodels"] = pd.Series(p_holm_sm, index=readout.index).loc[order]
holm.style.format({"p_raw": "{:.3e}", "p_times_multiplier": "{:.3e}", "p_holm": "{:.3e}", "p_holm_statsmodels": "{:.3e}"})

,comparison,p_raw,multiplier,p_times_multiplier,p_holm,p_holm_statsmodels
0,Mens E-Mail vs No E-Mail,1.469e-13,3,4.408e-13,4.408e-13,4.408e-13
1,Womens E-Mail vs No E-Mail,1.558e-04,2,3.115e-04,3.115e-04,3.115e-04
2,Mens E-Mail vs Womens E-Mail,2.056e-04,1,2.056e-04,3.115e-04,3.115e-04


## 5. Statistical and practical significance

Applying the pre-registered bars to each comparison.
- Statistical, Holm adjusted p below 0.05
- Practical, the estimated difference at least the MDE of 0.5 percentage points

For the two email vs No E-Mail comparisons these feed the ship tiers. Mens vs Womens only matters as the tiebreaker if both emails clear both bars.

In [9]:
readout["p_holm"] = holm["p_holm"].reindex(readout.index)
readout["stat_sig"] = readout["p_holm"] < ALPHA
readout["meets_mde"] = readout["diff_pp"].abs() >= MDE * 100
readout["ci_9833_excludes_zero"] = (readout["wald_lo_9833_pp"] > 0) | (readout["wald_hi_9833_pp"] < 0)

readout[["comparison", "diff_pp", "p_holm", "stat_sig", "meets_mde", "ci_9833_excludes_zero"]].style.format(
    {"diff_pp": "{:.4f}", "p_holm": "{:.3e}"}
)

,comparison,diff_pp,p_holm,stat_sig,meets_mde,ci_9833_excludes_zero
0,Mens E-Mail vs No E-Mail,0.6805,4.408e-13,True,True,True
1,Womens E-Mail vs No E-Mail,0.3111,3.115e-04,True,False,True
2,Mens E-Mail vs Womens E-Mail,0.3694,3.115e-04,True,False,True


In [10]:
readout.to_csv(TABLES / "conversion_readout.csv", index=False)
holm.to_csv(TABLES / "conversion_holm.csv", index=False)
by_arm.to_csv(TABLES / "conversion_by_arm.csv")
print("saved results/tables/conversion_readout.csv, conversion_holm.csv, conversion_by_arm.csv")

saved results/tables/conversion_readout.csv, conversion_holm.csv, conversion_by_arm.csv


## 6. Spend, pre-check. Does B = 5,000 hold for the BCa interval?

The spend CI is BCa (bias-corrected and accelerated, Efron 1987), with the percentile interval shown beside it. B = 5,000 was chosen from a convergence check on percentile CI width, so it needs checking for BCa too. Same setup as that check, pooled spend across all arms, seed 42, CI width at increasing B. Checked at both 95% and 98.33% since the spend CI level is not locked yet, and the further out the tails, the more resamples it takes to pin them down. Pooled only, no arm split.

**BCa, step by step.** With theta_hat the observed statistic and theta*_1..theta*_B the bootstrap replicates:

- bias correction, z0 = Phi^-1( share of replicates below theta_hat ). If the replicates sit centered on theta_hat, z0 = 0.
- acceleration, a = sum(d_i^3) / ( 6 x (sum(d_i^2))^(3/2) ), where d_i = mean of jackknife values - jackknife value i. Jackknife value i is the statistic recomputed with observation i left out. For a mean that is (total - x_i) / (n - 1), so no refitting is needed.
- adjusted percentiles, for each tail z = z_(alpha/2) or z_(1-alpha/2), use the percentile Phi( z0 + (z0 + z) / (1 - a(z0 + z)) ) of the replicates instead of alpha/2 and 1 - alpha/2.

With z0 = 0 and a = 0 this collapses back to the plain percentile interval.

In [11]:
def bca_interval(replicates, theta_hat, jack_values, alpha):
    z0 = stats.norm.ppf(np.mean(replicates < theta_hat))
    d = jack_values.mean() - jack_values
    a = np.sum(d ** 3) / (6 * np.sum(d ** 2) ** 1.5)
    z_lo, z_hi = stats.norm.ppf(alpha / 2), stats.norm.ppf(1 - alpha / 2)
    p_lo = stats.norm.cdf(z0 + (z0 + z_lo) / (1 - a * (z0 + z_lo)))
    p_hi = stats.norm.cdf(z0 + (z0 + z_hi) / (1 - a * (z0 + z_hi)))
    lo, hi = np.percentile(replicates, [100 * p_lo, 100 * p_hi])
    return lo, hi, z0, a, p_lo, p_hi

rng = np.random.default_rng(seed=42)
spend = df["spend"].to_numpy()
n = len(spend)
theta_hat = spend.mean()
jack_pooled = (spend.sum() - spend) / (n - 1)

max_b = 20000
boot_means = np.empty(max_b)
for b in range(max_b):
    boot_means[b] = spend[rng.integers(0, n, size=n)].mean()

rows = []
for level_name, lvl_alpha in [("95%", ALPHA), ("98.33%", ALPHA_ADJ)]:
    prev = None
    for b in [100, 500, 1000, 2000, 5000, 10000, 20000]:
        lo, hi, z0, a, p_lo, p_hi = bca_interval(boot_means[:b], theta_hat, jack_pooled, lvl_alpha)
        width = hi - lo
        rows.append({"level": level_name, "B": b, "z0": z0, "a": a, "p_lo": p_lo, "p_hi": p_hi,
                     "bca_lo": lo, "bca_hi": hi, "bca_width": width,
                     "pct_change_in_width_vs_prev": None if prev is None else (width / prev - 1) * 100})
        prev = width

bca_convergence = pd.DataFrame(rows)
print(f"pooled mean spend (theta_hat): {theta_hat:.6f}")
bca_convergence.round(6)

pooled mean spend (theta_hat): 1.050908


,level,B,z0,a,p_lo,p_hi,bca_lo,bca_hi,bca_width,pct_change_in_width_vs_prev
0,95%,100,-0.100434,0.013565,0.017654,0.964636,0.932251,1.140917,0.208667,NaN
1,95%,500,0.015040,0.013565,0.030062,0.979544,0.929854,1.154937,0.225083,7.867131
2,95%,1000,0.030084,0.013565,0.032116,0.981023,0.936301,1.166429,0.230128,2.241463
3,95%,2000,0.027576,0.013565,0.031766,0.980783,0.935789,1.168744,0.232954,1.228044
4,95%,5000,0.016043,0.013565,0.030195,0.979645,0.938757,1.172531,0.233774,0.352089
5,95%,10000,-0.003259,0.013565,0.027711,0.977614,0.939041,1.171255,0.232214,-0.667517
6,95%,20000,0.000501,0.013565,0.028181,0.978023,0.939011,1.171261,0.232250,0.015464
7,98.33%,100,-0.100434,0.013565,0.005982,0.988298,0.885886,1.154967,0.269081,NaN
8,98.33%,500,0.015040,0.013565,0.011024,0.993885,0.915033,1.174822,0.259789,-3.453062
9,98.33%,1000,0.030084,0.013565,0.011899,0.994403,0.915710,1.191879,0.276169,6.305158


In [12]:
bca_convergence.to_csv(TABLES / "spend_bca_convergence.csv", index=False)
print("saved results/tables/spend_bca_convergence.csv")

saved results/tables/spend_bca_convergence.csv


## 7. Spend, pre-check. What spend lift can the data detect?

Before spend is split by arm, the smallest difference in mean spend per customer the data could reliably detect, at 90% power (the locked power target), for the two candidate CI levels. Same relationship as the conversion power calc, with the pooled spend standard deviation sigma in place of the proportion variance terms:

detectable difference = (z_(alpha/2) + z_power) x sigma x sqrt(1/n_a + 1/n_b)

Approximation, stated up front. This assumes both arms share the pooled sigma and that the difference in means is close to normal. Spend is heavily skewed, so it is rougher than the conversion version, but with about 21,000 customers per arm the difference in means should still be close to normal. Pooled spend only, no arm split.

In [13]:
sigma_spend = df["spend"].std(ddof=1)
mean_spend = df["spend"].mean()
n_a, n_b = arm_sizes.nsmallest(2)
se_spend_diff = sigma_spend * np.sqrt(1 / n_a + 1 / n_b)
z_power = stats.norm.ppf(0.90)

print(f"pooled spend mean {mean_spend:.6f}, pooled spend SD {sigma_spend:.6f}")
print(f"two smallest arms {n_a:,} and {n_b:,}")
print(f"SE of a difference in mean spend {se_spend_diff:.6f}")
print(f"z_power (90%) {z_power:.4f}")

spend_power = []
for level_name, lvl_alpha in [("95%", ALPHA), ("98.33%", ALPHA_ADJ)]:
    z_a = stats.norm.ppf(1 - lvl_alpha / 2)
    detectable = (z_a + z_power) * se_spend_diff
    spend_power.append({"level": level_name, "z_alpha_half": z_a, "detectable_diff_dollars": detectable,
                        "detectable_relative_to_pooled_mean": detectable / mean_spend})
spend_power = pd.DataFrame(spend_power)
spend_power.to_csv(TABLES / "spend_detectable_diff.csv", index=False)
spend_power.style.format({"z_alpha_half": "{:.4f}", "detectable_diff_dollars": "${:.4f}", "detectable_relative_to_pooled_mean": "{:.2%}"})

pooled spend mean 1.050908, pooled spend SD 15.036448
two smallest arms 21,306 and 21,307
SE of a difference in mean spend 0.145681
z_power (90%) 1.2816


,level,z_alpha_half,detectable_diff_dollars,detectable_relative_to_pooled_mean
0,95%,1.9600,$0.4722,44.94%
1,98.33%,2.3940,$0.5355,50.95%


## 8. Spend, pre-check. How often could Welch and BCa disagree?

Spend counts as significant only if both methods agree, the Holm adjusted Welch p-value is below 0.05 and the 98.33% BCa interval excludes zero. If they disagree, the result is inconclusive. They can only disagree when the sampling distribution of mean spend is noticeably non-normal at arm size, because Welch assumes it is normal and BCa adjusts for skew. Earlier checks used all 64,000 customers, so this one checks at the size of a real arm. Pooled only, no arm labels used.

1. Draw a pseudo-arm of 21,306 customers (the smallest arm size) at random from the pooled data, seed 42.
2. Theoretical skewness of the sample mean, skew(mean) = skew(spend) / sqrt(n). 0 means a symmetric, normal-looking sampling distribution.
3. Bootstrap the pseudo-arm mean B = 5,000 times. Report the skewness of the replicates, plus z0 and a from BCa.
4. Compare three 98.33% intervals on the pseudo-arm mean.
   - normal theory, mean +/- z_crit x s / sqrt(n), the same normal approximation Welch's test uses
   - BCa
   - percentile
   For each, asymmetry = upper half-width / lower half-width (1 means symmetric). The gap between the normal and BCa lower bounds, in standard errors, is the zone where the two methods could disagree.

Why a single arm's mean is a worst case. Both arms' spend is right-skewed, so the difference of two arm means is less skewed than either arm's mean on its own (the skew partly cancels and the variances add). The real comparison should look at least this normal. The caveat is that the pseudo-arm shows the pooled spend shape, and a treated arm's tail could be somewhat heavier.

In [14]:
N_ARM = int(arm_sizes.min())
rng = np.random.default_rng(seed=42)
pseudo_arm = spend[rng.choice(n, size=N_ARM, replace=False)]
m = len(pseudo_arm)

mean_pa = pseudo_arm.mean()
sd_pa = pseudo_arm.std(ddof=1)
se_pa = sd_pa / np.sqrt(m)
skew_spend = stats.skew(pseudo_arm)
skew_mean_theory = skew_spend / np.sqrt(m)

print(f"pseudo-arm size {m:,}, share spending $0 {np.mean(pseudo_arm == 0):.4%}")
print(f"mean {mean_pa:.6f}, SD {sd_pa:.6f}, SE of mean {se_pa:.6f}")
print(f"skewness of spend {skew_spend:.4f}")
print(f"theoretical skewness of the mean, skew / sqrt(n) {skew_mean_theory:.6f}")

pseudo-arm size 21,306, share spending $0 99.1082%
mean 1.027391, SD 14.995884, SE of mean 0.102736
skewness of spend 21.5487
theoretical skewness of the mean, skew / sqrt(n) 0.147628


In [15]:
B = 5000
boot_pa = np.empty(B)
for b in range(B):
    boot_pa[b] = pseudo_arm[rng.integers(0, m, size=m)].mean()
jack_pa = (pseudo_arm.sum() - pseudo_arm) / (m - 1)

bca_lo, bca_hi, z0_pa, a_pa, p_lo_pa, p_hi_pa = bca_interval(boot_pa, mean_pa, jack_pa, ALPHA_ADJ)
pct_lo, pct_hi = np.percentile(boot_pa, [100 * ALPHA_ADJ / 2, 100 * (1 - ALPHA_ADJ / 2)])
norm_lo, norm_hi = mean_pa - z_crit_adj * se_pa, mean_pa + z_crit_adj * se_pa

print(f"bootstrap replicate skewness {stats.skew(boot_pa):.6f}")
print(f"bootstrap SD of replicates {boot_pa.std(ddof=1):.6f} vs analytic SE {se_pa:.6f}")
print(f"BCa z0 {z0_pa:.6f}, a {a_pa:.6f}, adjusted percentiles {p_lo_pa:.6f} / {p_hi_pa:.6f}")

intervals = []
for method, lo, hi in [("normal theory", norm_lo, norm_hi), ("BCa", bca_lo, bca_hi), ("percentile", pct_lo, pct_hi)]:
    intervals.append({"method": method, "lo": lo, "hi": hi, "width": hi - lo,
                      "lower_half": mean_pa - lo, "upper_half": hi - mean_pa,
                      "asymmetry_upper_over_lower": (hi - mean_pa) / (mean_pa - lo),
                      "lo_minus_normal_lo_in_se": (lo - norm_lo) / se_pa,
                      "hi_minus_normal_hi_in_se": (hi - norm_hi) / se_pa})
agreement = pd.DataFrame(intervals)
agreement.round(6)

bootstrap replicate skewness 0.112771
bootstrap SD of replicates 0.101702 vs analytic SE 0.102736
BCa z0 0.009525, a 0.024605, adjusted percentiles 0.012455 / 0.994828


,method,lo,hi,width,lower_half,upper_half,asymmetry_upper_over_lower,lo_minus_normal_lo_in_se,hi_minus_normal_hi_in_se
0,normal theory,0.781444,1.273338,0.491894,0.245947,0.245947,1.000000,0.000000,0.000000
1,BCa,0.809845,1.298926,0.489081,0.217546,0.271535,1.248174,0.276451,0.249066
2,percentile,0.797238,1.283095,0.485857,0.230153,0.255704,1.111020,0.153737,0.094974


In [16]:
agreement["pseudo_arm_n"] = m
agreement["skew_spend"] = skew_spend
agreement["skew_mean_theory"] = skew_mean_theory
agreement["skew_boot_replicates"] = stats.skew(boot_pa)
agreement["bca_z0"] = z0_pa
agreement["bca_a"] = a_pa
agreement.to_csv(TABLES / "spend_method_agreement.csv", index=False)
print("saved results/tables/spend_method_agreement.csv")

saved results/tables/spend_method_agreement.csv


## 9. Spend readout

Spend is now split by arm, which the 2026-10-18 note in `docs/experiment_design.md` allows (committed and pushed as 6f52a21 before this section was run). Methods, all locked in that note:
- Welch's t-test on mean spend per customer, for the same three comparisons as conversion
- its own Holm family of 3, significant if the Holm adjusted p is below 0.05
- BCa bootstrap interval as the main interval, 5,000 resamples, 98.33% level, percentile interval beside it, 95% as context
- a spend win needs Welch and BCa to agree, otherwise the result is inconclusive

### Spend by arm

For each arm, the customer count n, mean spend, the sample variance s^2, and the variance of the mean s^2 / n, which is what goes into the standard error.

In [17]:
spend_by_arm = df.groupby("segment")["spend"].agg(
    n="count", mean="mean", var=lambda s: s.var(ddof=1), share_zero=lambda s: (s == 0).mean(), max="max"
)
spend_by_arm["sd"] = np.sqrt(spend_by_arm["var"])
spend_by_arm["var_of_mean"] = spend_by_arm["var"] / spend_by_arm["n"]
spend_by_arm.round(6)

,n,mean,var,share_zero,max,sd,var_of_mean
segment,,,,,,,
Mens E-Mail,21307,1.422617,315.211799,0.987469,499.0,17.754205,0.014794
No E-Mail,21306,0.652789,134.286372,0.994274,499.0,11.588200,0.006303
Womens E-Mail,21387,1.077202,228.496675,0.991163,499.0,15.116106,0.010684


### Welch's t-test, worked by hand once, Mens vs No E-Mail

Welch's test does not assume the two arms have the same variance, which matters here since an email can change how spread out spend is.

difference = mean_t - mean_c

v_t = s_t^2 / n_t and v_c = s_c^2 / n_c, the variance of each arm's mean

SE = sqrt(v_t + v_c)

degrees of freedom (Welch-Satterthwaite) = (v_t + v_c)^2 / ( v_t^2 / (n_t - 1) + v_c^2 / (n_c - 1) )

t = difference / SE, and the two-sided p-value = 2 x P(T_df > |t|)

Cross-checked against scipy's `ttest_ind(equal_var=False)`.

In [18]:
arm_spend = {arm: g.to_numpy() for arm, g in df.groupby("segment")["spend"]}

def welch(x_t, x_c):
    v_t, v_c = x_t.var(ddof=1) / len(x_t), x_c.var(ddof=1) / len(x_c)
    diff = x_t.mean() - x_c.mean()
    se = np.sqrt(v_t + v_c)
    dof = (v_t + v_c) ** 2 / (v_t ** 2 / (len(x_t) - 1) + v_c ** 2 / (len(x_c) - 1))
    t_stat = diff / se
    return {"diff": diff, "v_t": v_t, "v_c": v_c, "se": se, "dof": dof, "t": t_stat,
            "p_raw": 2 * stats.t.sf(abs(t_stat), dof)}

w = welch(arm_spend["Mens E-Mail"], arm_spend["No E-Mail"])
scipy_w = stats.ttest_ind(arm_spend["Mens E-Mail"], arm_spend["No E-Mail"], equal_var=False)

print(f"mean_t {arm_spend['Mens E-Mail'].mean():.6f}, mean_c {arm_spend['No E-Mail'].mean():.6f}")
print(f"difference ${w['diff']:.6f} per customer")
print(f"v_t {w['v_t']:.6e}, v_c {w['v_c']:.6e}")
print(f"SE {w['se']:.6f}")
print(f"Welch degrees of freedom {w['dof']:.2f}")
print(f"t {w['t']:.4f}")
print(f"raw two-sided p {w['p_raw']:.6e}")
print(f"scipy check, t {scipy_w.statistic:.4f}, p {scipy_w.pvalue:.6e}")

mean_t 1.422617, mean_c 0.652789
difference $0.769827 per customer
v_t 1.479381e-02, v_c 6.302749e-03
SE 0.145247
Welch degrees of freedom 36671.16
t 5.3001
raw two-sided p 1.163815e-07
scipy check, t 5.3001, p 1.163815e-07


### Bootstrap intervals for a difference in mean spend

Each bootstrap replicate resamples each arm on its own, with replacement and at its own size, then takes the difference in means. That keeps the two arms independent, the way they were randomized. 5,000 replicates per comparison, one generator seeded at 42.

BCa uses the same steps as section 6. The only new piece is the jackknife for a two-arm statistic. Leaving out one treated customer i gives mean_t without i minus mean_c. Leaving out one control customer j gives mean_t minus mean_c without j. Putting all of those leave-one-out values together, the average is exactly the observed difference, so the `bca_interval` function from section 6 works unchanged. Each d_i works out to (x_i - mean_t) / (n_t - 1) for treated customers and -(x_j - mean_c) / (n_c - 1) for control customers.

The Welch t interval, difference +/- t_crit(df) x SE, is shown beside them so the normal approximation and the bootstrap can be compared directly.

In [19]:
B_SPEND = 5000
rng_spend = np.random.default_rng(seed=42)

rows = []
boot_store, jack_store = {}, {}
for arm_t, arm_c in comparisons:
    x_t, x_c = arm_spend[arm_t], arm_spend[arm_c]
    n_t, n_c = len(x_t), len(x_c)
    w = welch(x_t, x_c)

    boot = np.empty(B_SPEND)
    for b in range(B_SPEND):
        boot[b] = x_t[rng_spend.integers(0, n_t, size=n_t)].mean() - x_c[rng_spend.integers(0, n_c, size=n_c)].mean()
    jack = np.concatenate([(x_t.sum() - x_t) / (n_t - 1) - x_c.mean(), x_t.mean() - (x_c.sum() - x_c) / (n_c - 1)])

    boot_store[f"{arm_t} vs {arm_c}"], jack_store[f"{arm_t} vs {arm_c}"] = boot, jack
    row = {"comparison": f"{arm_t} vs {arm_c}", "mean_t": x_t.mean(), "mean_c": x_c.mean(), **w,
           "relative_lift": w["diff"] / x_c.mean(), "boot_sd": boot.std(ddof=1)}
    for label, lvl_alpha in [("9833", ALPHA_ADJ), ("95", ALPHA)]:
        lo, hi, z0, a, _, _ = bca_interval(boot, w["diff"], jack, lvl_alpha)
        p_lo, p_hi = np.percentile(boot, [100 * lvl_alpha / 2, 100 * (1 - lvl_alpha / 2)])
        t_crit = stats.t.ppf(1 - lvl_alpha / 2, w["dof"])
        row.update({f"bca_lo_{label}": lo, f"bca_hi_{label}": hi, f"pct_lo_{label}": p_lo, f"pct_hi_{label}": p_hi,
                    f"welch_lo_{label}": w["diff"] - t_crit * w["se"], f"welch_hi_{label}": w["diff"] + t_crit * w["se"]})
    row.update({"bca_z0": z0, "bca_a": a})
    rows.append(row)

spend_readout = pd.DataFrame(rows)
spend_readout[["comparison", "mean_t", "mean_c", "diff", "relative_lift", "se", "boot_sd", "dof", "t", "p_raw", "bca_z0", "bca_a"]].round(6)

,comparison,mean_t,mean_c,diff,relative_lift,se,boot_sd,dof,t,p_raw,bca_z0,bca_a
0,Mens E-Mail vs No E-Mail,1.422617,0.652789,0.769827,1.179289,0.145247,0.144291,36671.157977,5.300140,0.000000,-0.003008,0.007141
1,Womens E-Mail vs No E-Mail,1.077202,0.652789,0.424412,0.650152,0.130333,0.132798,40064.884407,3.256372,0.001129,-0.000501,0.004410
2,Mens E-Mail vs Womens E-Mail,1.422617,1.077202,0.345415,0.320660,0.159617,0.157879,41584.588483,2.164018,0.030469,0.004512,0.002980


In [20]:
spend_readout[["comparison", "welch_lo_9833", "welch_hi_9833", "bca_lo_9833", "bca_hi_9833", "pct_lo_9833", "pct_hi_9833"]].round(4)

,comparison,welch_lo_9833,welch_hi_9833,bca_lo_9833,bca_hi_9833,pct_lo_9833,pct_hi_9833
0,Mens E-Mail vs No E-Mail,0.4221,1.1176,0.4174,1.1061,0.4132,1.1034
1,Womens E-Mail vs No E-Mail,0.1124,0.7364,0.1243,0.7461,0.1215,0.7453
2,Mens E-Mail vs Womens E-Mail,-0.0367,0.7276,-0.0306,0.7242,-0.0327,0.7194


In [21]:
spend_readout[["comparison", "welch_lo_95", "welch_hi_95", "bca_lo_95", "bca_hi_95", "pct_lo_95", "pct_hi_95"]].round(4)

,comparison,welch_lo_95,welch_hi_95,bca_lo_95,bca_hi_95,pct_lo_95,pct_hi_95
0,Mens E-Mail vs No E-Mail,0.4851,1.0545,0.4839,1.0523,0.4791,1.0492
1,Womens E-Mail vs No E-Mail,0.1690,0.6799,0.1663,0.6962,0.1657,0.6942
2,Mens E-Mail vs Womens E-Mail,0.0326,0.6583,0.0467,0.6577,0.0375,0.6522


### Holm correction on the Welch p-values, by hand

Same steps as section 4, on the spend family of 3. Sort the raw p-values, multiply the k-th smallest by (3 - k + 1), take the running maximum, cap at 1. Cross-checked against statsmodels.

In [22]:
order_s = spend_readout["p_raw"].sort_values().index
spend_holm = pd.DataFrame({"comparison": spend_readout.loc[order_s, "comparison"], "p_raw": spend_readout.loc[order_s, "p_raw"]})
spend_holm["multiplier"] = [N_COMPARISONS - k for k in range(N_COMPARISONS)]
spend_holm["p_times_multiplier"] = spend_holm["p_raw"] * spend_holm["multiplier"]
spend_holm["p_holm"] = spend_holm["p_times_multiplier"].cummax().clip(upper=1)

_, p_holm_sm_s, _, _ = multipletests(spend_readout["p_raw"], alpha=ALPHA, method="holm")
spend_holm["p_holm_statsmodels"] = pd.Series(p_holm_sm_s, index=spend_readout.index).loc[order_s]
spend_holm.style.format({"p_raw": "{:.4e}", "p_times_multiplier": "{:.4e}", "p_holm": "{:.4e}", "p_holm_statsmodels": "{:.4e}"})

,comparison,p_raw,multiplier,p_times_multiplier,p_holm,p_holm_statsmodels
0,Mens E-Mail vs No E-Mail,1.1638e-07,3,3.4914e-07,3.4914e-07,3.4914e-07
1,Womens E-Mail vs No E-Mail,1.1294e-03,2,2.2588e-03,2.2588e-03,2.2588e-03
2,Mens E-Mail vs Womens E-Mail,3.0469e-02,1,3.0469e-02,3.0469e-02,3.0469e-02


### Applying the spend rule as first written

This is the rule as first written, with every interval at 98.33%. Section 10 applies it with each interval matched to its Holm bar, which is the main result.

For each comparison:
- Welch says significant if the Holm adjusted p is below 0.05
- BCa says significant if the 98.33% BCa interval excludes zero
- both agree on significant, a significant spend difference (its sign gives the direction). Both agree on not significant, not significant. They disagree, inconclusive

Then the decision chain from the design doc. Mens already ships broadly on conversion. Womens ships broadly only with a significant positive spend lift over No E-Mail. If it has one, Mens vs Womens on spend decides which email goes out, and if that is not a significant spend difference, Mens ships.

In [23]:
spend_readout["p_holm"] = spend_holm["p_holm"].reindex(spend_readout.index)
spend_readout["welch_sig"] = spend_readout["p_holm"] < ALPHA
spend_readout["bca_sig"] = (spend_readout["bca_lo_9833"] > 0) | (spend_readout["bca_hi_9833"] < 0)
spend_readout["methods_agree"] = spend_readout["welch_sig"] == spend_readout["bca_sig"]
spend_readout["spend_result"] = np.select(
    [~spend_readout["methods_agree"], spend_readout["welch_sig"] & (spend_readout["diff"] > 0),
     spend_readout["welch_sig"] & (spend_readout["diff"] < 0)],
    ["inconclusive, methods disagree", "significant, positive", "significant, negative"],
    default="not significant",
)
spend_readout[["comparison", "diff", "p_holm", "welch_sig", "bca_lo_9833", "bca_hi_9833", "bca_sig", "methods_agree", "spend_result"]].style.format(
    {"diff": "{:.4f}", "p_holm": "{:.4e}", "bca_lo_9833": "{:.4f}", "bca_hi_9833": "{:.4f}"}
)

,comparison,diff,p_holm,welch_sig,bca_lo_9833,bca_hi_9833,bca_sig,methods_agree,spend_result
0,Mens E-Mail vs No E-Mail,0.7698,3.4914e-07,True,0.4174,1.1061,True,True,"significant, positive"
1,Womens E-Mail vs No E-Mail,0.4244,2.2588e-03,True,0.1243,0.7461,True,True,"significant, positive"
2,Mens E-Mail vs Womens E-Mail,0.3454,3.0469e-02,True,-0.0306,0.7242,False,False,"inconclusive, methods disagree"


In [24]:
res = spend_readout.set_index("comparison")["spend_result"]
womens_spend = res["Womens E-Mail vs No E-Mail"]
mens_vs_womens_spend = res["Mens E-Mail vs Womens E-Mail"]

print(f"Mens vs No E-Mail spend: {res['Mens E-Mail vs No E-Mail']} (Mens already in the ship broadly tier on conversion)")
print(f"Womens vs No E-Mail spend: {womens_spend}")
if womens_spend == "significant, positive":
    print("Womens is shippable, tiebreaker goes to Mens vs Womens on spend")
    print(f"Mens vs Womens spend: {mens_vs_womens_spend}")
    if mens_vs_womens_spend == "significant, negative":
        email_to_ship = "Womens E-Mail"
    else:
        email_to_ship = "Mens E-Mail"
        if mens_vs_womens_spend != "significant, positive":
            print("Mens vs Womens on spend is not a significant difference, fallback applies")
else:
    print("Womens does not show a significant positive spend lift, Womens does not ship broadly")
    email_to_ship = "Mens E-Mail"
print(f"Email that ships broadly under the locked rule: {email_to_ship}")

Mens vs No E-Mail spend: significant, positive (Mens already in the ship broadly tier on conversion)
Womens vs No E-Mail spend: significant, positive
Womens is shippable, tiebreaker goes to Mens vs Womens on spend
Mens vs Womens spend: inconclusive, methods disagree
Mens vs Womens on spend is not a significant difference, fallback applies
Email that ships broadly under the locked rule: Mens E-Mail


### Statistical vs practical significance

The rule sets no dollar bar for spend, so practical size is stated in plain dollars per customer and relative to the No E-Mail mean, next to the smallest lift the data could reliably detect at the 98.33% level (section 7). Where Welch and the bootstrap intervals differ, that is noted here too.

In [25]:
detectable_9833 = spend_power.loc[spend_power["level"] == "98.33%", "detectable_diff_dollars"].iloc[0]
for _, r in spend_readout.iterrows():
    print(f"{r['comparison']}")
    print(f"  difference ${r['diff']:.4f} per customer ({r['relative_lift']:.2%} relative to the comparison arm)")
    print(f"  98.33% BCa (${r['bca_lo_9833']:.4f}, ${r['bca_hi_9833']:.4f}), Welch (${r['welch_lo_9833']:.4f}, ${r['welch_hi_9833']:.4f})")
    print(f"  BCa bounds minus Welch bounds, in SE: lower {(r['bca_lo_9833'] - r['welch_lo_9833']) / r['se']:+.3f}, upper {(r['bca_hi_9833'] - r['welch_hi_9833']) / r['se']:+.3f}")
    print(f"  Holm p {r['p_holm']:.4e}, result: {r['spend_result']}")
    print(f"  difference vs detectable lift at 98.33% (${detectable_9833:.4f}): {r['diff'] / detectable_9833:.2f}x")

Mens E-Mail vs No E-Mail
  difference $0.7698 per customer (117.93% relative to the comparison arm)
  98.33% BCa ($0.4174, $1.1061), Welch ($0.4221, $1.1176)
  BCa bounds minus Welch bounds, in SE: lower -0.032, upper -0.079
  Holm p 3.4914e-07, result: significant, positive
  difference vs detectable lift at 98.33% ($0.5355): 1.44x
Womens E-Mail vs No E-Mail
  difference $0.4244 per customer (65.02% relative to the comparison arm)
  98.33% BCa ($0.1243, $0.7461), Welch ($0.1124, $0.7364)
  BCa bounds minus Welch bounds, in SE: lower +0.091, upper +0.074
  Holm p 2.2588e-03, result: significant, positive
  difference vs detectable lift at 98.33% ($0.5355): 0.79x
Mens E-Mail vs Womens E-Mail
  difference $0.3454 per customer (32.07% relative to the comparison arm)
  98.33% BCa ($-0.0306, $0.7242), Welch ($-0.0367, $0.7276)
  BCa bounds minus Welch bounds, in SE: lower +0.038, upper -0.021
  Holm p 3.0469e-02, result: inconclusive, methods disagree
  difference vs detectable lift at 98.3

In [26]:
spend_by_arm.to_csv(TABLES / "spend_by_arm.csv")
spend_readout.to_csv(TABLES / "spend_readout.csv", index=False)
spend_holm.to_csv(TABLES / "spend_holm.csv", index=False)
print("saved results/tables/spend_by_arm.csv, spend_readout.csv, spend_holm.csv")

saved results/tables/spend_by_arm.csv, spend_readout.csv, spend_holm.csv


## 10. Spend rule with Holm rank-matched intervals (main result)

Holm does not use one bar for every comparison. It sorts the m = 3 p-values and compares the k-th smallest to alpha / (m - k + 1), so 0.05/3, then 0.05/2, then 0.05. It also stops at the first comparison that fails, and everything after that fails too. Section 9 held every interval at 98.33% (the 0.05/3 bar). That made the two halves of the rule disagree for the comparison ranked third, which Holm tests at 0.05.

Here each comparison's interval uses the same bar Holm uses for it. The Holm p is below 0.05 exactly when the raw p is below that bar and every earlier comparison passed, so the interval and the p-value now ask the same question. Welch and BCa can still disagree on how they handle skew, and the result is still inconclusive if they do.

| Holm rank | bar | interval level |
|---|---|---|
| 1 | 0.05 / 3 | 98.33% |
| 2 | 0.05 / 2 | 97.5% |
| 3 | 0.05 / 1 | 95% |

The same bootstrap replicates and jackknife values from section 9 are reused, so only the interval level changes.

In [27]:
sr = spend_readout.copy()
sr["holm_rank"] = sr["p_raw"].rank(method="first").astype(int)
sr["holm_bar"] = ALPHA / (N_COMPARISONS - sr["holm_rank"] + 1)
sr["matched_level"] = 1 - sr["holm_bar"]

for i, r in sr.iterrows():
    lo, hi, _, _, _, _ = bca_interval(boot_store[r["comparison"]], r["diff"], jack_store[r["comparison"]], r["holm_bar"])
    t_crit = stats.t.ppf(1 - r["holm_bar"] / 2, r["dof"])
    sr.loc[i, ["bca_lo_matched", "bca_hi_matched"]] = lo, hi
    sr.loc[i, ["welch_lo_matched", "welch_hi_matched"]] = r["diff"] - t_crit * r["se"], r["diff"] + t_crit * r["se"]

sr = sr.sort_values("holm_rank")
sr["bca_excludes_zero_matched"] = (sr["bca_lo_matched"] > 0) | (sr["bca_hi_matched"] < 0)
sr["bca_sig_matched"] = sr["bca_excludes_zero_matched"].astype(int).cummin().astype(bool)
sr["methods_agree_matched"] = sr["welch_sig"] == sr["bca_sig_matched"]
sr["spend_result_matched"] = np.select(
    [~sr["methods_agree_matched"], sr["welch_sig"] & (sr["diff"] > 0), sr["welch_sig"] & (sr["diff"] < 0)],
    ["inconclusive, methods disagree", "significant, positive", "significant, negative"],
    default="not significant",
)
sr[["comparison", "holm_rank", "holm_bar", "matched_level", "p_raw", "p_holm", "welch_sig",
    "welch_lo_matched", "welch_hi_matched", "bca_lo_matched", "bca_hi_matched", "bca_sig_matched", "spend_result_matched"]].style.format(
    {"holm_bar": "{:.5f}", "matched_level": "{:.2%}", "p_raw": "{:.4e}", "p_holm": "{:.4e}",
     "welch_lo_matched": "{:.4f}", "welch_hi_matched": "{:.4f}", "bca_lo_matched": "{:.4f}", "bca_hi_matched": "{:.4f}"}
)

,comparison,holm_rank,holm_bar,matched_level,p_raw,p_holm,welch_sig,welch_lo_matched,welch_hi_matched,bca_lo_matched,bca_hi_matched,bca_sig_matched,spend_result_matched
0,Mens E-Mail vs No E-Mail,1,0.01667,98.33%,1.1638e-07,3.4914e-07,True,0.4221,1.1176,0.4174,1.1061,True,"significant, positive"
1,Womens E-Mail vs No E-Mail,2,0.02500,97.50%,1.1294e-03,2.2588e-03,True,0.1323,0.7166,0.1389,0.7296,True,"significant, positive"
2,Mens E-Mail vs Womens E-Mail,3,0.05000,95.00%,3.0469e-02,3.0469e-02,True,0.0326,0.6583,0.0467,0.6577,True,"significant, positive"


In [28]:
res_m = sr.set_index("comparison")["spend_result_matched"]
womens_m = res_m["Womens E-Mail vs No E-Mail"]
mvw_m = res_m["Mens E-Mail vs Womens E-Mail"]

print(f"Mens vs No E-Mail spend: {res_m['Mens E-Mail vs No E-Mail']} (Mens already in the ship broadly tier on conversion)")
print(f"Womens vs No E-Mail spend: {womens_m}")
if womens_m == "significant, positive":
    print("Womens is shippable, tiebreaker goes to Mens vs Womens on spend")
    print(f"Mens vs Womens spend: {mvw_m}")
    if mvw_m == "significant, negative":
        email_to_ship_m = "Womens E-Mail"
    else:
        email_to_ship_m = "Mens E-Mail"
        if mvw_m != "significant, positive":
            print("Mens vs Womens on spend is not a significant difference, fallback applies")
else:
    print("Womens does not show a significant positive spend lift, Womens does not ship broadly")
    email_to_ship_m = "Mens E-Mail"
print(f"Email that ships broadly, rank-matched rule (main): {email_to_ship_m}")
print(f"Email that ships broadly, rule as first written (section 9): {email_to_ship}")

Mens vs No E-Mail spend: significant, positive (Mens already in the ship broadly tier on conversion)
Womens vs No E-Mail spend: significant, positive
Womens is shippable, tiebreaker goes to Mens vs Womens on spend
Mens vs Womens spend: significant, positive
Email that ships broadly, rank-matched rule (main): Mens E-Mail
Email that ships broadly, rule as first written (section 9): Mens E-Mail


In [29]:
compare_rules = sr[["comparison", "spend_result_matched"]].merge(
    spend_readout[["comparison", "spend_result"]].rename(columns={"spend_result": "spend_result_first_written"}), on="comparison")
print(compare_rules.to_string(index=False))

sr.to_csv(TABLES / "spend_rank_matched.csv", index=False)
print()
print("saved results/tables/spend_rank_matched.csv")

                  comparison  spend_result_matched     spend_result_first_written
    Mens E-Mail vs No E-Mail significant, positive          significant, positive
  Womens E-Mail vs No E-Mail significant, positive          significant, positive
Mens E-Mail vs Womens E-Mail significant, positive inconclusive, methods disagree

saved results/tables/spend_rank_matched.csv


## 11. Seed stability and the 50,000 resample spend intervals

Locked in `docs/experiment_design.md` (commit 0006737) before anything in this section was run. Sections 9 and 10 used one seed (42) and 5,000 resamples. A different seed draws different bootstrap samples, so the interval endpoints move a little. That movement is noise from the bootstrap itself, not from the data. The section 6 convergence check only measured interval width, not where the endpoints sit, so this section measures the endpoints directly.

Plan, as locked:
- 50 seeds (1 to 50) at 5,000 resamples, to measure how much the endpoints of the locked setting move
- the reported spend intervals rerun at 50,000 resamples, seed 42, whatever they show, with the 5,000 results kept beside them
- 10 seeds (1 to 10) at 50,000 to confirm the noise shrank as predicted
- this is a robustness view, not a gate. If any seed at 50,000 flips a bound across zero, the result is reported as not robust across seeds, and the locked result is not changed

Two formulas set the plan.

Bootstrap noise in an endpoint shrinks with 1 / sqrt(B). Going from 5,000 to 50,000 resamples is 10x, so the endpoint SD should shrink by sqrt(10), about 3.16x.

The SD across seeds is itself an estimate. Its standard error is about SD / sqrt(2 (n - 1)) for n seeds, so 50 seeds pin it to about +/-10% and 10 seeds to about +/-24%. That is why 50 seeds are used to measure the noise and 10 are enough to confirm a prediction.

Holm ranks, bars, and the Welch side of the rule come from the Welch p-values, which do not depend on the bootstrap, so they are taken from section 10 unchanged. Only the BCa intervals are recomputed.

In [30]:
for n_seeds in [10, 20, 50]:
    print(f"{n_seeds} seeds, relative SE of the SD estimate about {1 / np.sqrt(2 * (n_seeds - 1)):.1%}")
print(f"predicted shrink in endpoint SD from 5,000 to 50,000 resamples: sqrt(10) = {np.sqrt(10):.4f}x")

10 seeds, relative SE of the SD estimate about 23.6%
20 seeds, relative SE of the SD estimate about 16.2%
50 seeds, relative SE of the SD estimate about 10.1%
predicted shrink in endpoint SD from 5,000 to 50,000 resamples: sqrt(10) = 3.1623x


### Bootstrap runner

Same resampling as section 9. One generator per seed, the three comparisons in the same order, and inside each replicate the treated arm is drawn before the control arm. With seed 42 and 5,000 resamples this reproduces the section 9 replicates exactly, which is checked below.

For each comparison and seed it keeps the BCa interval at the Holm-matched level (main) and at 98.33% (the rule as first written).

In [31]:
holm_info = sr.set_index("comparison")[["holm_rank", "holm_bar", "matched_level", "diff", "welch_sig", "p_holm"]]

def run_boot(seed, B):
    rng_s = np.random.default_rng(seed=seed)
    out = {}
    for arm_t, arm_c in comparisons:
        x_t, x_c = arm_spend[arm_t], arm_spend[arm_c]
        n_t, n_c = len(x_t), len(x_c)
        boot = np.empty(B)
        for b in range(B):
            boot[b] = x_t[rng_s.integers(0, n_t, size=n_t)].mean() - x_c[rng_s.integers(0, n_c, size=n_c)].mean()
        out[f"{arm_t} vs {arm_c}"] = boot
    return out

def intervals_from(boots, seed, B):
    rows = []
    for comp, boot in boots.items():
        h = holm_info.loc[comp]
        lo_m, hi_m, *_ = bca_interval(boot, h["diff"], jack_store[comp], h["holm_bar"])
        lo_f, hi_f, *_ = bca_interval(boot, h["diff"], jack_store[comp], ALPHA_ADJ)
        rows.append({"seed": seed, "B": B, "comparison": comp, "matched_level": h["matched_level"],
                     "bca_lo_matched": lo_m, "bca_hi_matched": hi_m, "bca_lo_9833": lo_f, "bca_hi_9833": hi_f})
    return rows

check = run_boot(42, B_SPEND)
assert all(np.array_equal(check[k], boot_store[k]) for k in boot_store)
print("seed 42 at 5,000 resamples reproduces the section 9 replicates exactly")

seed 42 at 5,000 resamples reproduces the section 9 replicates exactly


### 50 seeds at 5,000 resamples

For each comparison and each bound, the min, median, max, and SD across seeds, and the share of seeds where the interval excludes zero.

In [32]:
rows_5k = []
for seed in range(1, 51):
    rows_5k += intervals_from(run_boot(seed, B_SPEND), seed, B_SPEND)
seeds_5k = pd.DataFrame(rows_5k)

def summarize(seeds_df):
    out = []
    for comp, g in seeds_df.groupby("comparison", sort=False):
        ex_m = ((g["bca_lo_matched"] > 0) | (g["bca_hi_matched"] < 0)).mean()
        ex_f = ((g["bca_lo_9833"] > 0) | (g["bca_hi_9833"] < 0)).mean()
        for col, ex in [("bca_lo_matched", ex_m), ("bca_hi_matched", ex_m), ("bca_lo_9833", ex_f), ("bca_hi_9833", ex_f)]:
            out.append({"comparison": comp, "bound": col, "min": g[col].min(), "median": g[col].median(),
                        "max": g[col].max(), "sd": g[col].std(ddof=1), "share_excluding_zero": ex})
    return pd.DataFrame(out)

summary_5k = summarize(seeds_5k)
summary_5k.style.format({c: "{:.4f}" for c in ["min", "median", "max", "sd"]} | {"share_excluding_zero": "{:.0%}"})

,comparison,bound,min,median,max,sd,share_excluding_zero
0,Mens E-Mail vs No E-Mail,bca_lo_matched,0.4130,0.4337,0.4550,0.0102,100%
1,Mens E-Mail vs No E-Mail,bca_hi_matched,1.1061,1.1278,1.1473,0.0089,100%
2,Mens E-Mail vs No E-Mail,bca_lo_9833,0.4130,0.4337,0.4550,0.0102,100%
3,Mens E-Mail vs No E-Mail,bca_hi_9833,1.1061,1.1278,1.1473,0.0089,100%
4,Womens E-Mail vs No E-Mail,bca_lo_matched,0.1149,0.1382,0.1534,0.0076,100%
5,Womens E-Mail vs No E-Mail,bca_hi_matched,0.6980,0.7205,0.7461,0.0080,100%
6,Womens E-Mail vs No E-Mail,bca_lo_9833,0.0924,0.1178,0.1311,0.0078,100%
7,Womens E-Mail vs No E-Mail,bca_hi_9833,0.7156,0.7444,0.7768,0.0097,100%
8,Mens E-Mail vs Womens E-Mail,bca_lo_matched,0.0156,0.0370,0.0571,0.0080,100%
9,Mens E-Mail vs Womens E-Mail,bca_hi_matched,0.6493,0.6617,0.6783,0.0072,100%


### Reported intervals, 50,000 resamples, seed 42

The locked main result. Same BCa method and Holm-matched levels as section 10, with the 5,000 resample intervals from section 10 beside them.

In [33]:
B_MAIN = 50000
main_50k = pd.DataFrame(intervals_from(run_boot(42, B_MAIN), 42, B_MAIN)).set_index("comparison")
main_50k = main_50k.join(sr.set_index("comparison")[["bca_lo_matched", "bca_hi_matched"]].rename(
    columns={"bca_lo_matched": "bca_lo_matched_5000", "bca_hi_matched": "bca_hi_matched_5000"}))
main_50k = main_50k.join(holm_info[["holm_rank", "diff", "p_holm", "welch_sig"]]).sort_values("holm_rank")
main_50k[["holm_rank", "matched_level", "diff", "bca_lo_matched", "bca_hi_matched", "bca_lo_matched_5000",
          "bca_hi_matched_5000", "bca_lo_9833", "bca_hi_9833"]].style.format(
    {"matched_level": "{:.2%}"} | {c: "{:.4f}" for c in ["diff", "bca_lo_matched", "bca_hi_matched", "bca_lo_matched_5000",
                                                         "bca_hi_matched_5000", "bca_lo_9833", "bca_hi_9833"]})

,holm_rank,matched_level,diff,bca_lo_matched,bca_hi_matched,bca_lo_matched_5000,bca_hi_matched_5000,bca_lo_9833,bca_hi_9833
comparison,,,,,,,,,
Mens E-Mail vs No E-Mail,1,98.33%,0.7698,0.4355,1.1266,0.4174,1.1061,0.4355,1.1266
Womens E-Mail vs No E-Mail,2,97.50%,0.4244,0.1358,0.7213,0.1389,0.7296,0.1153,0.7420
Mens E-Mail vs Womens E-Mail,3,95.00%,0.3454,0.0328,0.6601,0.0467,0.6577,-0.0329,0.7296


### Spend rule applied at 50,000 resamples

Same rule as section 10. BCa calls a comparison significant if its matched interval excludes zero, and like Holm it stops at the first comparison in rank order that fails. Welch and BCa have to agree, otherwise the result is inconclusive. Then the same decision chain. The rule as first written (every interval at 98.33%) is applied beside it.

In [34]:
def spend_results(df_, lo, hi, holm_stop):
    excl = (df_[lo] > 0) | (df_[hi] < 0)
    bca_sig = excl.astype(int).cummin().astype(bool) if holm_stop else excl
    agree = df_["welch_sig"] == bca_sig
    return pd.Series(np.select(
        [~agree, df_["welch_sig"] & (df_["diff"] > 0), df_["welch_sig"] & (df_["diff"] < 0)],
        ["inconclusive, methods disagree", "significant, positive", "significant, negative"], default="not significant"),
        index=df_.index)

def email_to_ship_from(res_):
    if res_["Womens E-Mail vs No E-Mail"] != "significant, positive":
        return "Mens E-Mail", "Womens has no significant positive spend lift"
    mvw = res_["Mens E-Mail vs Womens E-Mail"]
    if mvw == "significant, negative":
        return "Womens E-Mail", "Womens wins the tiebreaker on spend"
    if mvw == "significant, positive":
        return "Mens E-Mail", "Mens wins the tiebreaker on spend"
    return "Mens E-Mail", "tiebreaker on spend not significant, fallback applies"

main_50k["spend_result_matched"] = spend_results(main_50k, "bca_lo_matched", "bca_hi_matched", holm_stop=True)
main_50k["spend_result_9833"] = spend_results(main_50k, "bca_lo_9833", "bca_hi_9833", holm_stop=False)
print(main_50k[["spend_result_matched", "spend_result_9833"]].to_string())
print()
for label, col in [("rank-matched (main)", "spend_result_matched"), ("as first written", "spend_result_9833")]:
    email, why = email_to_ship_from(main_50k[col])
    print(f"50,000 resamples, {label}: {email} ships broadly ({why})")

                               spend_result_matched               spend_result_9833
comparison                                                                         
Mens E-Mail vs No E-Mail      significant, positive           significant, positive
Womens E-Mail vs No E-Mail    significant, positive           significant, positive
Mens E-Mail vs Womens E-Mail  significant, positive  inconclusive, methods disagree

50,000 resamples, rank-matched (main): Mens E-Mail ships broadly (Mens wins the tiebreaker on spend)
50,000 resamples, as first written: Mens E-Mail ships broadly (tiebreaker on spend not significant, fallback applies)


### 10 seeds at 50,000 resamples, did the noise shrink as predicted?

Predicted endpoint SD at 50,000 = SD at 5,000 (from the 50 seeds) / sqrt(10). The observed SD from 10 seeds is compared to it. With 10 seeds the observed SD is only good to about +/-24%, so a ratio anywhere near 1 confirms the prediction.

In [35]:
rows_50k = []
for seed in range(1, 11):
    rows_50k += intervals_from(run_boot(seed, B_MAIN), seed, B_MAIN)
seeds_50k = pd.DataFrame(rows_50k)
summary_50k = summarize(seeds_50k)

shrink = summary_5k[["comparison", "bound", "sd"]].merge(
    summary_50k[["comparison", "bound", "sd"]], on=["comparison", "bound"], suffixes=("_5000", "_50000"))
shrink["predicted_sd_50000"] = shrink["sd_5000"] / np.sqrt(10)
shrink["observed_over_predicted"] = shrink["sd_50000"] / shrink["predicted_sd_50000"]
shrink.style.format({c: "{:.4f}" for c in ["sd_5000", "sd_50000", "predicted_sd_50000"]} | {"observed_over_predicted": "{:.2f}"})

,comparison,bound,sd_5000,sd_50000,predicted_sd_50000,observed_over_predicted
0,Mens E-Mail vs No E-Mail,bca_lo_matched,0.0102,0.0029,0.0032,0.91
1,Mens E-Mail vs No E-Mail,bca_hi_matched,0.0089,0.0022,0.0028,0.80
2,Mens E-Mail vs No E-Mail,bca_lo_9833,0.0102,0.0029,0.0032,0.91
3,Mens E-Mail vs No E-Mail,bca_hi_9833,0.0089,0.0022,0.0028,0.80
4,Womens E-Mail vs No E-Mail,bca_lo_matched,0.0076,0.0014,0.0024,0.59
5,Womens E-Mail vs No E-Mail,bca_hi_matched,0.0080,0.0021,0.0025,0.82
6,Womens E-Mail vs No E-Mail,bca_lo_9833,0.0078,0.0015,0.0025,0.62
7,Womens E-Mail vs No E-Mail,bca_hi_9833,0.0097,0.0021,0.0031,0.69
8,Mens E-Mail vs Womens E-Mail,bca_lo_matched,0.0080,0.0027,0.0025,1.05
9,Mens E-Mail vs Womens E-Mail,bca_hi_matched,0.0072,0.0024,0.0023,1.04


In [36]:
summary_50k.style.format({c: "{:.4f}" for c in ["min", "median", "max", "sd"]} | {"share_excluding_zero": "{:.0%}"})

,comparison,bound,min,median,max,sd,share_excluding_zero
0,Mens E-Mail vs No E-Mail,bca_lo_matched,0.4276,0.4330,0.4368,0.0029,100%
1,Mens E-Mail vs No E-Mail,bca_hi_matched,1.1264,1.1306,1.1350,0.0022,100%
2,Mens E-Mail vs No E-Mail,bca_lo_9833,0.4276,0.4330,0.4368,0.0029,100%
3,Mens E-Mail vs No E-Mail,bca_hi_9833,1.1264,1.1306,1.1350,0.0022,100%
4,Womens E-Mail vs No E-Mail,bca_lo_matched,0.1363,0.1377,0.1403,0.0014,100%
5,Womens E-Mail vs No E-Mail,bca_hi_matched,0.7201,0.7225,0.7260,0.0021,100%
6,Womens E-Mail vs No E-Mail,bca_lo_9833,0.1159,0.1193,0.1213,0.0015,100%
7,Womens E-Mail vs No E-Mail,bca_hi_9833,0.7405,0.7432,0.7471,0.0021,100%
8,Mens E-Mail vs Womens E-Mail,bca_lo_matched,0.0319,0.0358,0.0404,0.0027,100%
9,Mens E-Mail vs Womens E-Mail,bca_hi_matched,0.6614,0.6628,0.6686,0.0024,100%


### Robustness verdict

For each comparison, does any seed land on the other side of zero from the seed 42 result at 50,000? If any seed at 50,000 does, the result is reported as not robust across seeds. The same count is shown for the 50 seeds at 5,000, as context on the locked setting.

In [37]:
for comp in main_50k.index:
    main_excl = bool((main_50k.loc[comp, "bca_lo_matched"] > 0) or (main_50k.loc[comp, "bca_hi_matched"] < 0))
    for label, sd_ in [("50,000", seeds_50k), ("5,000", seeds_5k)]:
        g = sd_[sd_["comparison"] == comp]
        excl = (g["bca_lo_matched"] > 0) | (g["bca_hi_matched"] < 0)
        flips = int((excl != main_excl).sum())
        print(f"{comp}, matched level, {label} resamples: seed 42 at 50,000 excludes zero {main_excl}, seeds that flip {flips} of {len(g)}")
    print()

Mens E-Mail vs No E-Mail, matched level, 50,000 resamples: seed 42 at 50,000 excludes zero True, seeds that flip 0 of 10
Mens E-Mail vs No E-Mail, matched level, 5,000 resamples: seed 42 at 50,000 excludes zero True, seeds that flip 0 of 50

Womens E-Mail vs No E-Mail, matched level, 50,000 resamples: seed 42 at 50,000 excludes zero True, seeds that flip 0 of 10
Womens E-Mail vs No E-Mail, matched level, 5,000 resamples: seed 42 at 50,000 excludes zero True, seeds that flip 0 of 50

Mens E-Mail vs Womens E-Mail, matched level, 50,000 resamples: seed 42 at 50,000 excludes zero True, seeds that flip 0 of 10
Mens E-Mail vs Womens E-Mail, matched level, 5,000 resamples: seed 42 at 50,000 excludes zero True, seeds that flip 0 of 50



In [38]:
seeds_5k.to_csv(TABLES / "spend_seed_stability_5000.csv", index=False)
seeds_50k.to_csv(TABLES / "spend_seed_stability_50000.csv", index=False)
pd.concat([summary_5k.assign(B=5000), summary_50k.assign(B=50000)]).to_csv(TABLES / "spend_seed_stability_summary.csv", index=False)
shrink.to_csv(TABLES / "spend_seed_noise_shrink.csv", index=False)
main_50k.reset_index().to_csv(TABLES / "spend_readout_b50000.csv", index=False)
print("saved spend_seed_stability_5000.csv, spend_seed_stability_50000.csv, spend_seed_stability_summary.csv,")
print("      spend_seed_noise_shrink.csv, spend_readout_b50000.csv")

saved spend_seed_stability_5000.csv, spend_seed_stability_50000.csv, spend_seed_stability_summary.csv,
      spend_seed_noise_shrink.csv, spend_readout_b50000.csv


## 12. Dollar impact for the decision memo

Scope as locked in `docs/experiment_design.md`. One send, the two week window, revenue not profit. Nothing is annualized and no margin or email cost is applied, since the data has none of those.

incremental revenue = spend lift per customer x number of customers

The lift and its interval are the reported 50,000 resample results from section 11, at the Holm-matched level. Multiplying by a fixed count scales the bounds the same way, so the interval carries through unchanged in shape.

The number of customers is the experiment's own customer base, the 64,000 rows. For any other list of N customers like these, the same formula applies with N in place of 64,000, and a per 10,000 figure is shown for that.

- Mens vs No E-Mail is the headline, since the decision is Mens against sending nothing
- Mens vs Womens is what is gained by sending Mens instead of Womens

In [39]:
N_CUSTOMERS = len(df)
impact = main_50k.loc[["Mens E-Mail vs No E-Mail", "Mens E-Mail vs Womens E-Mail"],
                      ["matched_level", "diff", "bca_lo_matched", "bca_hi_matched"]].copy()
for col, name in [("diff", "point"), ("bca_lo_matched", "lo"), ("bca_hi_matched", "hi")]:
    impact[f"per_10000_{name}"] = impact[col] * 10000
    impact[f"total_{name}"] = impact[col] * N_CUSTOMERS
print(f"customers in the experiment: {N_CUSTOMERS:,}")
for comp, r in impact.iterrows():
    print(f"{comp}, {r['matched_level']:.2%} BCa, 50,000 resamples")
    print(f"  per customer ${r['diff']:.4f} (${r['bca_lo_matched']:.4f}, ${r['bca_hi_matched']:.4f})")
    print(f"  per 10,000 customers ${r['per_10000_point']:,.2f} (${r['per_10000_lo']:,.2f}, ${r['per_10000_hi']:,.2f})")
    print(f"  x {N_CUSTOMERS:,} customers ${r['total_point']:,.2f} (${r['total_lo']:,.2f}, ${r['total_hi']:,.2f})")

customers in the experiment: 64,000
Mens E-Mail vs No E-Mail, 98.33% BCa, 50,000 resamples
  per customer $0.7698 ($0.4355, $1.1266)
  per 10,000 customers $7,698.27 ($4,354.51, $11,265.81)
  x 64,000 customers $49,268.94 ($27,868.89, $72,101.18)
Mens E-Mail vs Womens E-Mail, 95.00% BCa, 50,000 resamples
  per customer $0.3454 ($0.0328, $0.6601)
  per 10,000 customers $3,454.15 ($328.26, $6,600.71)
  x 64,000 customers $22,106.56 ($2,100.89, $42,244.56)


In [40]:
impact.reset_index().to_csv(TABLES / "dollar_impact.csv", index=False)
print("saved results/tables/dollar_impact.csv")

saved results/tables/dollar_impact.csv
